<a href="https://colab.research.google.com/github/scottn66/255-final-project/blob/main/hyb20_lab3_parallel_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — parallel warm-started fine-tunes (one GPU, several variants)

Runs several variants of the Lab 3 network at the same time on one GPU with `lab3_finetune.py`. Every variant
starts from the finished 11-block run (`lab3_best_b11.pt`); blocks added beyond 11 start as an exact identity, so
no variant can start below the source model. Each variant writes into its own folder,
`Drive/.../lab3/runs/<tag>/`, so nothing is overwritten.

Run the cells top to bottom on a GPU runtime. The monitor cell keeps the runtime busy and prints every
evaluation line; the last cell ranks the finished variants by float PSNR on `valid/`.

Depth and latency (estimates from the NPU timings; only a timed `.mxq` settles it): 11 blocks is at the 12 ms
edge, and each extra block adds about 0.75 ms. Lab 3 bands: under 12 ms = 18 points, under 14 = 17, under 16 = 16,
all only if the NPU PSNR is above 26.2 dB.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, zipfile
D = '/content/drive/MyDrive/Colab Notebooks/DATA 255'
KIT = f'{D}/lab2/colab_compile_kit.zip'
assert os.path.exists(KIT), f'kit missing: {KIT}'
shutil.rmtree('/content/kit', ignore_errors=True)
with zipfile.ZipFile(KIT) as z:
    z.extractall('/content/kit')
print(sorted(os.listdir('/content/kit')))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['calib_images', 'compile_probes.sh', 'convert_mxq.py', 'model_deploy_reluclamp.onnx', 'probes', 'qubee-0.12.0.0+aries2-py3-none-any.whl']


In [ ]:
%%bash
set -e
sed -i 's/\r$//' /content/kit/compile_probes.sh /content/kit/convert_mxq.py
pip install -q uv
uv venv --python 3.10 --seed /content/venv310
PY=/content/venv310/bin/python
$PY -m pip install -q torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu128
$PY -m pip install -q /content/kit/qubee-0.12.0.0+aries2-py3-none-any.whl numpy==1.26.0 onnx==1.16.2 onnxruntime==1.19.2 \
  tensorflow==2.17.0 opencv-python-headless pillow tqdm scipy pandas PyYAML msgpack pycocotools typeguard
$PY - <<'EOF'
import torch, qubee
from qubee import mmc
print('torch', torch.__version__, '| qubee', getattr(qubee, '__version__', '?'), '| mmc hw type:', mmc.Compiler().get_hw_type())
EOF

use reshape_check_util with py backend.
torch 2.7.1+cu128 | qubee 0.12.0.0 | mmc hw type: Aries2


Using CPython 3.10.22
Creating virtual environment with seed packages at: venv310
 + packaging==26.3
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source venv310/bin/activate
[2026-11-11 07:39:41] qubee.mobilint_transformers.compat_transformers : [INFO] The transformers library isn’t installed. Please verify the transformers version in the qubee docker image and install it.
[2026-11-11 07:39:43] qubee.model_dict.parser.backend.torch.parser : [INFO] The transformers library isn’t installed. Please verify which transformers version is compatible with the Qubee Docker image and install that specific version to ensure proper model parsing.


In [ ]:
%%bash
D="/content/drive/MyDrive/Colab Notebooks/DATA 255"
sed 's/\r$//' "$D/lab2/lab3_compile_variant.sh" > /content/lab3_compile_variant.sh
if [ "$(nproc)" -ge 4 ]; then
  for t in b13_psnr b15_psnr b12_psnr; do bash /content/lab3_compile_variant.sh $t "" 0.99 > /content/compile_$t.log 2>&1 & done; wait
  cat /content/compile_b13_psnr.log /content/compile_b15_psnr.log /content/compile_b12_psnr.log
else
  for t in b13_psnr b15_psnr b12_psnr; do bash /content/lab3_compile_variant.sh $t "" 0.99; done
fi
ls -la "$D/lab3/sweep_mxq/" | grep -E "b1[235]_psnr"

[b13_psnr] percentile 0.99 | zip /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs/b13_psnr/b13_psnr_npu.zip | onnx 1939441 bytes | calib 100
[b13_psnr] float_reference: {"tag":"b13_psnr","n_blocks":13,"trained_3x3_convs":15,"loss":"psnr","valid_psnr_float":26.269376514563284,"identity_valid_psnr":25.12128167605756,"selection_psnr":27.306171477203968,"best_step":18000,"is_ema":true,"warm_start_selection_psnr":27.251131255103218,"fuse_max_diff":7.152557373046875e-07,
clamped ops {'Conv': 17, 'Relu': 16, 'Add': 1} | noclamp ops {'Conv': 15, 'Relu': 14, 'Add': 1}
max|clamped - clip(noclamp)| 2.98e-08 | raw range -0.250 .. 1.124
[b13_psnr] b13_psnr_p99_clamped exit 0 wrote mxq/b13_psnr_p99_clamped.mxq (1.62 MB), calibrated on 100 images from calib_images
[b13_psnr] b13_psnr_p99_noclamp exit 0 wrote mxq/b13_psnr_p99_noclamp.mxq (1.61 MB), calibrated on 100 images from calib_images
-rw------- 1 root root 1621409 Nov 11 07:43 /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/sweep_

In [ ]:
# Setup: Drive, data, the two scripts, the source checkpoint.
import os, shutil, sys, torch
WORK = '/content'
DRIVE_LAB3 = '/content/drive/MyDrive/Colab Notebooks/DATA 255/lab3'
DATA_ROOT = '/content/lab3_data'
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(f'{DATA_ROOT}/train'):
        os.makedirs(DATA_ROOT, exist_ok=True)
        for z in ('train.zip', 'valid.zip'):
            get_ipython().system(f'unzip -q -o "{DRIVE_LAB3}/{z}" -d {DATA_ROOT}')
    get_ipython().system('pip -q install onnx onnxruntime')
    get_ipython().system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
for f in ('lab3_finetune.py', 'lab3_model.py'):
    shutil.copy(f'{DRIVE_LAB3}/{f}', f'{WORK}/{f}')
SRC = f'{WORK}/lab3_best_b11.pt'
shutil.copy(f'{DRIVE_LAB3}/lab3_best_b11.pt', SRC)
_s = torch.load(SRC, map_location='cpu')
print(f"source checkpoint: step {_s['step']}, selection PSNR {_s['sel_psnr']:.4f}, "
      f"{'EMA' if _s['is_ema'] else 'raw'} weights (expect step 80000, 27.2511)")
OUT = f'{DRIVE_LAB3}/runs'
os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive
NVIDIA L4, 23034 MiB
source checkpoint: step 80000, selection PSNR 27.2511, EMA weights (expect step 80000, 27.2511)


In [ ]:
import subprocess, shutil
shutil.copy(f'{DRIVE_LAB3}/lab3_half.py', f'{WORK}/lab3_half.py')
# Session 2:
RUNS = [dict(tag='half30b', script='lab3_half.py', args=['--blocks', '30', '--lr1', '1e-4'])]
# Session 3 (use this line instead):
# RUNS = [dict(tag='half20', script='lab3_half.py', args=['--blocks', '20']),
#         dict(tag='b13s_psnr', script='lab3_finetune.py', args=['--blocks', '13', '--loss', 'psnr',
#              '--src', f'{DRIVE_LAB3}/runs/b13_scratch/lab3_best.pt'])]
procs = globals().get('procs', {})
for r in RUNS:
    if r['tag'] in procs and procs[r['tag']].poll() is None:
        print(f"{r['tag']}: already running"); continue
    cmd = [sys.executable, '-u', r['script'], '--tag', r['tag'], '--data-root', DATA_ROOT, '--out', OUT] + r['args']
    procs[r['tag']] = subprocess.Popen(cmd, cwd=WORK, stdout=open(f"{WORK}/{r['tag']}.log", 'a'),
                                       stderr=subprocess.STDOUT, start_new_session=True)
    print(f"{r['tag']}: started (pid {procs[r['tag']].pid})")

half30b: started (pid 9949)


In [ ]:
# Launch a hybrid run (own session, so a cell interrupt can't reach it; re-running resumes from its checkpoint).
import subprocess, shutil
shutil.copy(f'{DRIVE_LAB3}/lab3_half_v2.py', f'{WORK}/lab3_half_v2.py')
RUNS = [dict(tag='hyb16', blocks=20)]          # for the 16-block one: dict(tag='hyb16', blocks=16)
procs = globals().get('procs', {})
for r in RUNS:
    if r['tag'] in procs and procs[r['tag']].poll() is None:
        print(f"{r['tag']}: already running"); continue
    cmd = [sys.executable, '-u', 'lab3_half_v2.py', '--tag', r['tag'], '--arch', 'hybrid',
           '--blocks', str(r['blocks']), '--data-root', DATA_ROOT, '--out', OUT]
    procs[r['tag']] = subprocess.Popen(cmd, cwd=WORK, stdout=open(f"{WORK}/{r['tag']}.log", 'a'),
                                       stderr=subprocess.STDOUT, start_new_session=True)
    print(f"{r['tag']}: started (pid {procs[r['tag']].pid})")

hyb16: already running


In [ ]:
# procs['b12_mse'].terminate(); del procs['b12_mse']
procs['half20'].terminate(); del procs['half30b']

In [ ]:
!ps -eo pid,pgid,etime,args | grep "tag hyb20" | grep -v grep

  12580   12580    02:40:46 /usr/bin/python3 -u lab3_half_v2.py --tag hyb20 --arch hybrid --blocks 20 --data-root /content/lab3_data --out /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs
  12646   12580    02:40:35 /usr/bin/python3 -u lab3_half_v2.py --tag hyb20 --arch hybrid --blocks 20 --data-root /content/lab3_data --out /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs
  12647   12580    02:40:35 /usr/bin/python3 -u lab3_half_v2.py --tag hyb20 --arch hybrid --blocks 20 --data-root /content/lab3_data --out /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs


In [ ]:
# Monitor: prints each run's progress lines as they appear and blocks until every run has finished.
# Interrupting this cell does NOT stop the runs; re-run it to keep watching. To stop one: procs['tag'].terminate()
import time, subprocess
POLL = 30
pos = globals().get('pos', {})
while True:
    for tag, p in procs.items():
        with open(f'{WORK}/{tag}.log') as f:
            f.seek(pos.get(tag, 0)); chunk = f.read(); pos[tag] = f.tell()
        for line in chunk.splitlines():
            if line.startswith('['):
                print(line)
    if all(p.poll() is not None for p in procs.values()):
        break
    time.sleep(POLL)
for tag, p in procs.items():
    if p.returncode != 0:
        print(f'\n=== {tag} FAILED (exit {p.returncode}); last lines of its log:')
        print(''.join(open(f'{WORK}/{tag}.log').readlines()[-15:]))
        print('If this is CUDA out of memory: lower BATCH (or run fewer variants at once) and re-run the launch cell.')
print('\nall runs finished:', {t: p.returncode for t, p in procs.items()})

[hyb20] stage 1 step  56000/90000 | loss 0.0314 | selection raw 27.2887 / ema 27.3291  <- new best | valid(ema, monitor) 26.3128 | 6.3 it/s, eta 90 min | gpu 2.1 GB
[hyb20] stage 1 step  58000/90000 | loss 0.0437 | selection raw 27.2966 / ema 27.3327  <- new best | valid(ema, monitor) 26.3150 | 6.3 it/s, eta 84 min | gpu 2.1 GB
[hyb20] stage 1 step  60000/90000 | loss 0.0362 | selection raw 27.3228 / ema 27.3335  <- new best | valid(ema, monitor) 26.3196 | 6.3 it/s, eta 79 min | gpu 2.1 GB
[hyb20] stage 1 step  62000/90000 | loss 0.0343 | selection raw 27.3244 / ema 27.3343  <- new best | valid(ema, monitor) 26.3198 | 6.3 it/s, eta 74 min | gpu 2.1 GB
[hyb16] arch hybrid, 20 body convs at 128x128 | stage 1 90000 steps, stage 2 20000 | batch 16 | device cuda | out /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs/hyb16
[hyb16] identity at init ok | fuse max diff (fp64) 2.6e-13 | random-weight graph /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs/hyb16/hyb16_random.onnx

In [ ]:
# Leaderboard: every finished variant in OUT, best float PSNR on valid/ first.
# Float must be about 26.32 or more to expect >= 26.25 on the NPU (the 11-block model lost 0.066 dB to int8).
import glob, json
rows = [json.load(open(p)) for p in glob.glob(f'{OUT}/*/float_reference.json')]
print(f"{'tag':12s} {'blocks':>6s} {'loss':>5s} {'valid float':>12s} {'vs b11':>8s} {'selection':>10s} {'best step':>10s} {'est. grader ms':>15s}")
for r in sorted(rows, key=lambda r: -r['valid_psnr_float']):
    est = 12.0 + 0.75 * (r['n_blocks'] - 11)
    print(f"{r['tag']:12s} {r['n_blocks']:6d} {r['loss']:>5s} {r['valid_psnr_float']:12.4f} "
          f"{r['valid_psnr_float'] - 26.2189:+8.4f} {r['selection_psnr']:10.4f} {r['best_step']:10d} {est:11.1f} +-0.3")
print('\n11-block source model: valid float 26.2189, NPU 26.1527. Each <tag>_npu.zip is in', OUT)

tag          blocks  loss  valid float   vs b11  selection  best step  est. grader ms
b15_psnr         15  psnr      26.2828  +0.0639    27.3192      20000        15.0 +-0.3
b13_psnr         13  psnr      26.2694  +0.0505    27.3062      18000        13.5 +-0.3
b12_psnr         12  psnr      26.2586  +0.0397    27.2989      19000        12.8 +-0.3

11-block source model: valid float 26.2189, NPU 26.1527. Each <tag>_npu.zip is in /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs


In [ ]:
%%bash
D="/content/drive/MyDrive/Colab Notebooks/DATA 255"
test -x /content/venv310/bin/python && test -f /content/kit/convert_mxq.py || { echo "Compile runtime was reset: run the compile notebook's setup cells first"; exit 1; }
sed 's/\r$//' "$D/lab2/lab3_compile_variant.sh" > /content/lab3_compile_variant.sh
for t in b13_psnr b15_psnr b12_psnr; do bash /content/lab3_compile_variant.sh $t "" 0.99; done

Compile runtime was reset: run the compile notebook's setup cells first


CalledProcessError: Command 'b'D="/content/drive/MyDrive/Colab Notebooks/DATA 255"\ntest -x /content/venv310/bin/python && test -f /content/kit/convert_mxq.py || { echo "Compile runtime was reset: run the compile notebook\'s setup cells first"; exit 1; }\nsed \'s/\\r$//\' "$D/lab2/lab3_compile_variant.sh" > /content/lab3_compile_variant.sh\nfor t in b13_psnr b15_psnr b12_psnr; do bash /content/lab3_compile_variant.sh $t "" 0.99; done\n'' returned non-zero exit status 1.

In [ ]:
%%bash
set -u
D="/content/drive/MyDrive/Colab Notebooks/DATA 255"
sed 's/\r$//' "$D/lab2/convert_mxq_sweep.py" > /content/convert_mxq_sweep.py
for tag in b12_psnr b13_psnr b12_mse; do
  W=/content/work_$tag; rm -rf $W; mkdir -p $W; cd $W
  unzip -q "$D/lab3/runs/$tag/${tag}_npu.zip"; cd lab3_npu; mkdir -p sweep_mxq
  for p in 0.999 0.9999; do n=lab3_${tag}_p${p#0.}
    /content/venv310/bin/python /content/convert_mxq_sweep.py --onnx lab3_deploy.onnx --calib-dir calib_images \
      --out sweep_mxq/$n.mxq --percentile $p > sweep_mxq/$n.log 2>&1
    echo "$n exit $? $(tail -n 1 sweep_mxq/$n.log)"; cp sweep_mxq/$n.mxq sweep_mxq/$n.log "$D/lab3/sweep_mxq/"
  done
done

lab3_b12_psnr_p999 exit 127 bash: line 8: /content/venv310/bin/python: No such file or directory
lab3_b12_psnr_p9999 exit 127 bash: line 8: /content/venv310/bin/python: No such file or directory
lab3_b13_psnr_p999 exit 127 bash: line 8: /content/venv310/bin/python: No such file or directory
lab3_b13_psnr_p9999 exit 127 bash: line 8: /content/venv310/bin/python: No such file or directory
lab3_b12_mse_p999 exit 127 bash: line 8: /content/venv310/bin/python: No such file or directory
lab3_b12_mse_p9999 exit 127 bash: line 8: /content/venv310/bin/python: No such file or directory


cp: cannot stat 'sweep_mxq/lab3_b12_psnr_p999.mxq': No such file or directory
cp: cannot stat 'sweep_mxq/lab3_b12_psnr_p9999.mxq': No such file or directory
cp: cannot stat 'sweep_mxq/lab3_b13_psnr_p999.mxq': No such file or directory
cp: cannot stat 'sweep_mxq/lab3_b13_psnr_p9999.mxq': No such file or directory
unzip:  cannot find or open /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs/b12_mse/b12_mse_npu.zip, /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs/b12_mse/b12_mse_npu.zip.zip or /content/drive/MyDrive/Colab Notebooks/DATA 255/lab3/runs/b12_mse/b12_mse_npu.zip.ZIP.
bash: line 6: cd: lab3_npu: No such file or directory
cp: cannot stat 'sweep_mxq/lab3_b12_mse_p999.mxq': No such file or directory
cp: cannot stat 'sweep_mxq/lab3_b12_mse_p9999.mxq': No such file or directory


CalledProcessError: Command 'b'set -u\nD="/content/drive/MyDrive/Colab Notebooks/DATA 255"\nsed \'s/\\r$//\' "$D/lab2/convert_mxq_sweep.py" > /content/convert_mxq_sweep.py\nfor tag in b12_psnr b13_psnr b12_mse; do\n  W=/content/work_$tag; rm -rf $W; mkdir -p $W; cd $W\n  unzip -q "$D/lab3/runs/$tag/${tag}_npu.zip"; cd lab3_npu; mkdir -p sweep_mxq\n  for p in 0.999 0.9999; do n=lab3_${tag}_p${p#0.}\n    /content/venv310/bin/python /content/convert_mxq_sweep.py --onnx lab3_deploy.onnx --calib-dir calib_images \\\n      --out sweep_mxq/$n.mxq --percentile $p > sweep_mxq/$n.log 2>&1\n    echo "$n exit $? $(tail -n 1 sweep_mxq/$n.log)"; cp sweep_mxq/$n.mxq sweep_mxq/$n.log "$D/lab3/sweep_mxq/"\n  done\ndone\n'' returned non-zero exit status 1.

## Next: compile and score

In the **compile notebook's runtime** (it has `/content/venv310` and `/content/kit`), paste this cell. Edit the
`for tag in ...` list to the variants worth scoring. About 5 minutes per file.

```bash
%%bash
set -u
D="/content/drive/MyDrive/Colab Notebooks/DATA 255"
sed 's/\r$//' "$D/lab2/convert_mxq_sweep.py" > /content/convert_mxq_sweep.py
for tag in b12_psnr b13_psnr b12_mse; do
  W=/content/work_$tag; rm -rf $W; mkdir -p $W; cd $W
  unzip -q "$D/lab3/runs/$tag/${tag}_npu.zip"; cd lab3_npu; mkdir -p sweep_mxq
  for p in 0.999 0.9999; do n=lab3_${tag}_p${p#0.}
    /content/venv310/bin/python /content/convert_mxq_sweep.py --onnx lab3_deploy.onnx --calib-dir calib_images \
      --out sweep_mxq/$n.mxq --percentile $p > sweep_mxq/$n.log 2>&1
    echo "$n exit $? $(tail -n 1 sweep_mxq/$n.log)"; cp sweep_mxq/$n.mxq sweep_mxq/$n.log "$D/lab3/sweep_mxq/"
  done
done
```

Then on the NPU PC, with the `.mxq` files in `results\`, one line scores them all:

```powershell
Get-ChildItem results\lab3_b*_p*.mxq | ForEach-Object { C:\NPU_Projects\venv312\Scripts\python.exe npu_psnr.py --mxq $_.FullName --valid-dir valid --out "results\npu_psnr_$($_.BaseName).json" }
```

Submit only a file that scores at least 26.25 dB on the "uint8-rounded output" line, and set `N_BLOCKS` in
`lab3_model.py` to that variant's depth so the architecture file matches the weights.